# 第16章: メトリック・セマンティックSLAM

本ノートブックでは、幾何情報とセマンティック（意味）情報を統合するSLAMについて学びます。

## 学習内容
1. **セマンティック占有格子地図**: 各セルに意味ラベルを付与した地図の構築
2. **セマンティック支援データアソシエーション**: 意味情報によるマッチング改善
3. **オブジェクトレベルSLAM**: バウンディングボックスによる物体レベルの地図表現

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
np.random.seed(42)

## 1. セマンティック占有格子地図

従来の占有格子地図は各セルが「占有/空き/未知」のみでしたが、セマンティック地図では各セルに意味ラベル（壁、家具、ドアなど）を付与します。これにより、ロボットはより高度なナビゲーションや意思決定が可能になります。

In [ ]:
# --- Semantic Occupancy Grid Map ---

# Semantic labels
EMPTY = 0
WALL = 1
FURNITURE = 2
DOOR = 3
WINDOW = 4

label_names = {EMPTY: 'Empty', WALL: 'Wall', FURNITURE: 'Furniture', DOOR: 'Door', WINDOW: 'Window'}
label_colors = ['white', '#404040', '#8B4513', '#228B22', '#4169E1']
cmap = ListedColormap(label_colors)

def create_semantic_room(size=40):
    """Create a room floor plan with semantic labels."""
    grid = np.zeros((size, size), dtype=int)
    
    # Outer walls
    grid[0, :] = WALL
    grid[-1, :] = WALL
    grid[:, 0] = WALL
    grid[:, -1] = WALL
    
    # Inner wall (room divider)
    grid[0:25, 20] = WALL
    grid[25:, 28] = WALL
    
    # Doors
    grid[22:25, 20] = DOOR  # Door in left room divider
    grid[25, 25:28] = DOOR  # Door in right section
    
    # Windows
    grid[0, 8:14] = WINDOW
    grid[0, 30:36] = WINDOW
    
    # Furniture (tables, chairs, shelves)
    # Table in left room
    grid[8:12, 5:10] = FURNITURE
    # Shelf along wall
    grid[2:8, 1:3] = FURNITURE
    # Table in right room
    grid[10:14, 25:30] = FURNITURE
    # Desk in bottom area
    grid[30:34, 8:14] = FURNITURE
    # Bookshelf
    grid[30:38, 30:32] = FURNITURE
    
    return grid

# Create and visualize
semantic_grid = create_semantic_room()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Semantic map
im = axes[0].imshow(semantic_grid, cmap=cmap, vmin=0, vmax=4, origin='upper')
axes[0].set_title('Semantic Occupancy Grid', fontsize=14, fontweight='bold')
axes[0].set_xlabel('x (grid cells)')
axes[0].set_ylabel('y (grid cells)')
# Legend
patches = [mpatches.Patch(color=label_colors[i], label=label_names[i]) for i in range(5)]
axes[0].legend(handles=patches, loc='lower right', fontsize=10)

# Traditional occupancy (binary)
binary_grid = (semantic_grid > 0).astype(int)
axes[1].imshow(binary_grid, cmap='gray_r', origin='upper')
axes[1].set_title('Traditional Occupancy Grid\n(No Semantic Info)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('x (grid cells)')
axes[1].set_ylabel('y (grid cells)')

plt.tight_layout()
plt.savefig('semantic_grid.png', dpi=100, bbox_inches='tight')
plt.show()
print("セマンティック地図は各セルの意味を保持し、ナビゲーション計画に活用できます")
print(f"例: ドアセル数={np.sum(semantic_grid == DOOR)}, 家具セル数={np.sum(semantic_grid == FURNITURE)}")

## 2. セマンティック支援データアソシエーション

データアソシエーション（観測と地図上のランドマークの対応付け）は、SLAMの重要なステップです。セマンティック情報を使うことで、異なるクラスの物体間の誤った対応を防ぎ、マッチング精度を向上させます。

In [ ]:
# --- Semantic-aided data association ---

def data_association_geometric(observations, landmarks, threshold=2.0):
    """Pure geometric data association: nearest neighbor by distance."""
    associations = []
    for i, obs in enumerate(observations):
        dists = np.linalg.norm(landmarks[:, :2] - obs[:2], axis=1)
        j = np.argmin(dists)
        if dists[j] < threshold:
            associations.append((i, j))
    return associations

def data_association_semantic(observations, landmarks, obs_labels, lm_labels, threshold=2.0):
    """Semantic-aided: match only within same semantic class."""
    associations = []
    for i, obs in enumerate(observations):
        # Only consider landmarks with matching semantic label
        same_class = lm_labels == obs_labels[i]
        if not np.any(same_class):
            continue
        candidates = np.where(same_class)[0]
        dists = np.linalg.norm(landmarks[candidates, :2] - obs[:2], axis=1)
        best = np.argmin(dists)
        if dists[best] < threshold:
            associations.append((i, candidates[best]))
    return associations

# Landmarks in map (x, y) with semantic labels
# 0=chair, 1=table, 2=door
landmark_positions = np.array([
    [3, 4], [5, 7], [8, 3], [2, 8], [7, 6], [4, 2], [9, 8], [6, 1],
], dtype=float)
landmark_labels = np.array([0, 1, 2, 0, 1, 0, 2, 1])  # chair, table, door, ...
label_map = {0: 'Chair', 1: 'Table', 2: 'Door'}
color_map = {0: 'orange', 1: 'blue', 2: 'green'}

# New observations (with noise, some ambiguous geometrically)
obs_positions = np.array([
    [3.2, 4.3],  # Should match landmark 0 (chair)
    [5.3, 6.8],  # Should match landmark 1 (table)
    [7.8, 3.2],  # Should match landmark 2 (door)
    [4.1, 2.3],  # Should match landmark 5 (chair)
], dtype=float)
obs_labels = np.array([0, 1, 2, 0])  # Known semantic labels from detection

# Ground truth associations
gt_associations = [(0, 0), (1, 1), (2, 2), (3, 5)]

# Compare methods
assoc_geo = data_association_geometric(obs_positions, landmark_positions)
assoc_sem = data_association_semantic(obs_positions, landmark_positions, obs_labels, landmark_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, assoc, title in [(axes[0], assoc_geo, 'Geometric Only'),
                          (axes[1], assoc_sem, 'Semantic-Aided')]:
    # Plot landmarks
    for k, (lm, lab) in enumerate(zip(landmark_positions, landmark_labels)):
        ax.scatter(lm[0], lm[1], c=color_map[lab], s=150, marker='s', zorder=5,
                   edgecolors='black', linewidth=1.5)
        ax.annotate(f'L{k}({label_map[lab]})', (lm[0]+0.15, lm[1]+0.25), fontsize=8)
    
    # Plot observations
    for k, (obs, lab) in enumerate(zip(obs_positions, obs_labels)):
        ax.scatter(obs[0], obs[1], c=color_map[lab], s=100, marker='*', zorder=5,
                   edgecolors='black', linewidth=1)
        ax.annotate(f'O{k}({label_map[lab]})', (obs[0]+0.15, obs[1]-0.4), fontsize=8)
    
    # Draw associations
    correct = 0
    for i, j in assoc:
        is_correct = (i, j) in gt_associations
        color = 'green' if is_correct else 'red'
        style = '-' if is_correct else '--'
        correct += int(is_correct)
        ax.plot([obs_positions[i, 0], landmark_positions[j, 0]],
                [obs_positions[i, 1], landmark_positions[j, 1]],
                color=color, linewidth=2, linestyle=style)
    
    ax.set_title(f'{title}\n(Correct: {correct}/{len(gt_associations)})', fontsize=13, fontweight='bold')
    ax.set_xlim(0, 11)
    ax.set_ylim(0, 10)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('semantic_association.png', dpi=100, bbox_inches='tight')
plt.show()
print("緑線=正しい対応, 赤破線=誤った対応")

## 3. オブジェクトレベルSLAM

ポイントレベルではなく、物体レベルでの地図表現を考えます。各物体はバウンディングボックスとセマンティックラベルで表現され、物体の検出・追跡・地図への統合が行われます。

In [ ]:
# --- Object-Level SLAM with Bounding Boxes ---

class ObjectMap:
    """Simple object-level map with bounding boxes."""
    def __init__(self):
        self.objects = []  # list of {center, size, label, confidence, observations}
    
    def add_or_update(self, center, size, label, merge_threshold=2.0):
        """Add new object or update existing one if close enough."""
        for obj in self.objects:
            if obj['label'] == label:
                dist = np.linalg.norm(np.array(obj['center']) - np.array(center))
                if dist < merge_threshold:
                    # Update: running average
                    n = obj['observations']
                    obj['center'] = [(obj['center'][k] * n + center[k]) / (n + 1) for k in range(2)]
                    obj['size'] = [(obj['size'][k] * n + size[k]) / (n + 1) for k in range(2)]
                    obj['observations'] += 1
                    obj['confidence'] = min(1.0, obj['confidence'] + 0.1)
                    return
        # New object
        self.objects.append({
            'center': list(center),
            'size': list(size),
            'label': label,
            'confidence': 0.5,
            'observations': 1
        })

# Simulate object detections from multiple camera poses
obj_map = ObjectMap()

# Simulated detections (from 3 different viewpoints with noise)
detections = [
    # Viewpoint 1
    [([3, 4], [2, 1.5], 'table'), ([7, 3], [1.5, 1.5], 'chair'), ([5, 8], [1, 2], 'door')],
    # Viewpoint 2 (noisy re-observations + new objects)
    [([3.2, 4.1], [2.1, 1.4], 'table'), ([7.1, 2.8], [1.6, 1.4], 'chair'), 
     ([5.1, 7.9], [0.9, 2.1], 'door'), ([9, 6], [2, 2], 'shelf')],
    # Viewpoint 3
    [([2.9, 3.9], [1.9, 1.6], 'table'), ([7.2, 3.1], [1.4, 1.6], 'chair'),
     ([9.1, 5.9], [2.1, 1.9], 'shelf'), ([1, 6], [1, 1], 'chair')],
]

for frame_detections in detections:
    for center, size, label in frame_detections:
        obj_map.add_or_update(center, size, label)

# Visualize object-level map
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

obj_colors = {'table': '#8B4513', 'chair': '#FF8C00', 'door': '#228B22', 'shelf': '#4169E1'}

for obj in obj_map.objects:
    cx, cy = obj['center']
    w, h = obj['size']
    color = obj_colors.get(obj['label'], 'gray')
    alpha = obj['confidence']
    
    # Draw bounding box
    rect = FancyBboxPatch((cx - w/2, cy - h/2), w, h,
                          boxstyle="round,pad=0.05",
                          facecolor=color, alpha=alpha * 0.4,
                          edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    
    # Label
    ax.text(cx, cy, f"{obj['label']}\nconf={obj['confidence']:.1f}\nobs={obj['observations']}",
            ha='center', va='center', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

# Draw robot trajectory
robot_positions = [[2, 1], [5, 1.5], [8, 1]]
ax.plot([p[0] for p in robot_positions], [p[1] for p in robot_positions],
        'k-o', markersize=10, linewidth=2, label='Robot trajectory')
ax.annotate('Start', robot_positions[0], fontsize=10, ha='center',
            xytext=(0, -15), textcoords='offset points')

ax.set_xlim(-1, 12)
ax.set_ylim(-1, 10)
ax.set_aspect('equal')
ax.set_title('Object-Level Semantic Map', fontsize=14, fontweight='bold')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('object_level_slam.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n地図上の物体数: {len(obj_map.objects)}")
for obj in obj_map.objects:
    print(f"  {obj['label']:8s}: center=({obj['center'][0]:.1f}, {obj['center'][1]:.1f}), "
          f"conf={obj['confidence']:.1f}, observations={obj['observations']}")

## 演習問題

1. **セマンティック地図の更新**: 複数のセンサー観測をベイズ的に統合して、各セルのセマンティックラベルの確率分布を維持する仕組みを実装してください。
2. **オブジェクトマージ閾値**: `merge_threshold` を変えて、物体の分離と統合のトレードオフを観察してください。
3. **セマンティックコスト地図**: セマンティックラベルに応じた通行コスト（壁=無限、家具=高、空き=低、ドア=低）を定義し、経路計画に活用してみてください。

## まとめ

- **セマンティック占有格子地図**は、幾何情報に意味ラベルを付加し、環境のより豊かな理解を提供する
- **セマンティック支援データアソシエーション**は、クラス情報を制約として利用し、誤対応を削減する
- **オブジェクトレベルSLAM**は、ポイントクラウドよりもコンパクトで人間にとって理解しやすい地図表現を提供
- 実際のシステムでは、セマンティックセグメンテーション（DeepLab等）や物体検出（YOLO等）のDNNと統合される